# Proyecto Final de IA: Predicción del Precio de Venta de Viviendas

## Descripción del Problema

El objetivo de este proyecto es construir un modelo de **regresión supervisada** capaz de predecir el precio de venta (`SalePrice`) de viviendas residenciales en Ames, Iowa (USA), a partir de 80 variables descriptivas de cada propiedad.

Este es un problema de **aprendizaje supervisado de regresión**: conocemos el precio real de cada vivienda en el conjunto de entrenamiento, y queremos que el modelo generalice esa relación para predecir precios de viviendas nuevas.

### Dataset: Ames Housing (House Prices)
- **Origen**: OpenML / Kaggle "House Prices: Advanced Regression Techniques"
- **Tamaño**: 1.460 viviendas × 81 variables
- **Variable objetivo**: `SalePrice` (precio de venta en USD)
- **Variables predictoras**: 79 features que incluyen área habitable, calidad de construcción, año de construcción, barrio, número de habitaciones, garaje, etc.

### Estructura del Proyecto
| Ejercicio | Contenido |
|-----------|-----------|
| **Ejercicio 1** | Análisis Exploratorio (EDA) y Preprocesamiento |
| **Ejercicio 2** | Modelado con Random Forest y Evaluación |
| **Ejercicio 3** | Optimización de Hiperparámetros, Comparación de Modelos y Despliegue |

> **Nota de ejecución**: Este notebook está diseñado para ejecutarse en Google Colab o VSCode (con extensión Jupyter). Cada celda puede ejecutarse de forma independiente y secuencial.

---
## EJERCICIO 1: Big Data — Preparación y Análisis

### Importación de Librerías

Importamos todas las librerías necesarias para el pipeline completo: análisis, visualización, preprocesamiento y modelado.

Aunque el dataset (~1.460 filas) no requiere herramientas de Big Data como `PySpark` o `Dask`, la arquitectura usada con `sklearn.pipeline` y `ColumnTransformer` escala bien a conjuntos de datos más grandes.

In [1]:
# --- Librerías de manipulación y análisis de datos ---
import pandas as pd          # DataFrames y operaciones tabulares
import numpy as np           # Operaciones numéricas y álgebra lineal

# --- Librerías de visualización ---
import matplotlib
matplotlib.use('Agg')        # Backend sin interfaz gráfica (necesario en servidores/Colab sin display)
import matplotlib.pyplot as plt
import seaborn as sns

# --- Utilidades ---
import time                  # Medir tiempos de entrenamiento
import joblib                # Serializar/deserializar modelos entrenados
import os

# Fix SSL para descarga del dataset desde OpenML (entornos con certificados no verificados)
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

# --- Carga del dataset ---
from sklearn.datasets import fetch_openml

# --- Preprocesamiento ---
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# --- Modelos de regresión ---
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor     # Red neuronal (para comparación opcional)

# --- Métricas de evaluación ---
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("Librerías importadas correctamente.")

Librerías importadas correctamente.


### Carga del Dataset

Usamos `fetch_openml` para descargar directamente el dataset "House Prices" de OpenML. El parámetro `parser='auto'` permite que pandas infiera los tipos de columna automáticamente.

In [2]:
print("Cargando dataset 'House Prices' desde OpenML...")
housing = fetch_openml(name="house_prices", as_frame=True, version=1, parser='auto')
df = housing.frame

print(f"\nDimensiones del dataset: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"Variable objetivo: 'SalePrice' (precio de venta en USD)")
print(f"\nPrimeras 5 filas del dataset:")
df.head()

Cargando dataset 'House Prices' desde OpenML...

Dimensiones del dataset: 1460 filas × 81 columnas
Variable objetivo: 'SalePrice' (precio de venta en USD)

Primeras 5 filas del dataset:


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


### Análisis Exploratorio de los Datos (EDA)

El EDA nos permite entender la estructura del dataset antes de modelar. Los objetivos son:
1. Detectar valores faltantes y su magnitud
2. Entender la distribución de la variable objetivo (`SalePrice`)
3. Identificar qué variables tienen mayor correlación con el precio
4. Detectar outliers y variables con distribución sesgada

In [ ]:
# --- Estadística Descriptiva ---
# describe() calcula automáticamente las medidas estadísticas básicas para variables numéricas:
# count (número de valores no nulos), mean, std, min, percentiles y max.
# Esto nos da una primera visión de escala, dispersión y posibles outliers.
print("Estadística Descriptiva (variables numéricas):")
print(df.describe().T.to_string())  # .T para leer mejor con muchas columnas

In [ ]:
# --- Análisis de Valores Faltantes (Nulos) ---
# Es fundamental saber qué columnas tienen datos faltantes y en qué proporción,
# ya que los modelos de sklearn no aceptan NaN directamente.
print("Columnas con valores nulos (solo las afectadas):\n")
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(2)
nulos_df = pd.DataFrame({'Nulos': nulos, 'Porcentaje (%)': nulos_pct})
nulos_df = nulos_df[nulos_df['Nulos'] > 0].sort_values('Porcentaje (%)', ascending=False)
print(nulos_df.to_string())
print(f"\nTotal columnas con nulos: {len(nulos_df)} de {df.shape[1]}")
print(f"Tipos de datos: {df.dtypes.value_counts().to_dict()}")

In [5]:
os.makedirs('visualization', exist_ok=True)
print("Carpeta 'visualization/' lista para guardar los gráficos.")

Carpeta 'visualization/' lista para guardar los gráficos.


In [ ]:
# --- Análisis de Correlación con SalePrice ---
# La correlación de Pearson mide la relación lineal entre dos variables (-1 a +1).
# Variables con alta correlación positiva suben junto al precio;
# con alta correlación negativa, suben cuando el precio baja.
# Solo aplica directamente a variables numéricas.
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()

top_features = corr_matrix['SalePrice'].sort_values(ascending=False).head(11)  # Top 10 + propia
print("Top 10 variables numéricas más correlacionadas con SalePrice:")
print(top_features.drop('SalePrice').to_string())

# Mapa de calor para visualizar correlaciones cruzadas entre las top variables
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Heatmap de correlaciones entre las top features
top_corr = numeric_df[top_features.index].corr()
sns.heatmap(top_corr, annot=True, cmap='coolwarm', fmt=".2f", ax=axes[0], square=True)
axes[0].set_title('Matriz de Correlación — Top 10 Variables Numéricas')

# Barplot de correlaciones con SalePrice
corr_con_precio = top_features.drop('SalePrice')
corr_con_precio.plot(kind='barh', ax=axes[1], color='steelblue', edgecolor='black')
axes[1].set_title('Correlación de Variables con SalePrice')
axes[1].set_xlabel('Coeficiente de Correlación de Pearson')
axes[1].axvline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.savefig('visualization/corr_matrix.png', dpi=100)
plt.close()

print("\nInterpretación:")
print("--> OverallQual (r=0.79) y GrLivArea (r=0.71) dominan la predicción del precio.")
print("--> GarageCars y GarageArea están muy correlacionadas entre sí (multicolinealidad).")
print("--> Los modelos de árbol manejan multicolinealidad mejor que la regresión lineal.")

In [ ]:
# --- Distribución de la Variable Objetivo (SalePrice) ---
# Analizar la distribución de SalePrice es clave porque:
# - Si está muy sesgada (cola larga a la derecha), los modelos lineales se verán afectados.
# - Un sesgo pronunciado sugiere aplicar una transformación logarítmica (log1p) para
#   normalizarla, aunque los modelos de árbol como Random Forest son robustos ante esto.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de SalePrice original
sns.histplot(df['SalePrice'], bins=50, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Distribución del Precio de Venta (SalePrice)')
axes[0].set_xlabel('SalePrice (USD)')
axes[0].set_ylabel('Frecuencia')
axes[0].axvline(df['SalePrice'].mean(), color='red', linestyle='--', label=f"Media: ${df['SalePrice'].mean():,.0f}")
axes[0].axvline(df['SalePrice'].median(), color='orange', linestyle='--', label=f"Mediana: ${df['SalePrice'].median():,.0f}")
axes[0].legend()

skewness = df['SalePrice'].skew()
print(f"Asimetría (Skewness) de SalePrice: {skewness:.4f}")
print("Un valor de skewness > 1 confirma sesgo positivo (cola a la derecha).")
print("La transformación log aplana la distribución y la acerca a una normal.")

if skewness > 1.0:
    # Histograma con transformación logarítmica
    normalized_df = np.log1p(df['SalePrice'])
    sns.histplot(normalized_df, bins=50, kde=True, ax=axes[1], color='seagreen')
    axes[1].set_title('Distribución de log(SalePrice+1) — Normalizada')
    axes[1].set_xlabel('log(SalePrice + 1)')
    axes[1].set_ylabel('Frecuencia')
    axes[1].axvline(normalized_df.mean(), color='red', linestyle='--', label=f"Media: ${normalized_df.mean():,.0f}")
    axes[1].axvline(normalized_df.median(), color='orange', linestyle='--', label=f"Mediana: ${normalized_df.median():,.0f}")

    plt.suptitle('Análisis de la Variable Objetivo: SalePrice', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('visualization/hist_saleprice.png', dpi=100)
plt.close()

In [ ]:
# --- Diagrama de Dispersión: Área Habitable vs Precio ---
# GrLivArea (superficie habitable sobre tierra en pies²) es la variable numérica
# con mayor correlación con SalePrice después de OverallQual.
# Un scatter plot nos revela:
# - Si la relación es lineal o no lineal
# - Si existen outliers extremos (p.ej. casas muy grandes vendidas a bajo precio)
plt.figure(figsize=(10, 5))
sns.scatterplot(x=df['GrLivArea'], y=df['SalePrice'], alpha=0.4, color='seagreen')
plt.title('Área Habitable vs Precio de Venta')
plt.xlabel('Área Habitable — GrLivArea (pies²)')
plt.ylabel('Precio de Venta — SalePrice (USD)')

# Marcar los dos outliers clásicos de este dataset (muy grande + precio bajo)
outliers = df[(df['GrLivArea'] > 4000) & (df['SalePrice'] < 300000)]
plt.scatter(outliers['GrLivArea'], outliers['SalePrice'], color='red', s=80, zorder=5, label='Posibles outliers')
plt.legend()
plt.tight_layout()
plt.savefig('visualization/scatter_livarea.png', dpi=100)
plt.close()

print("Correlación GrLivArea ↔ SalePrice:", round(df['GrLivArea'].corr(df['SalePrice']), 4))
print(f"Nota: los {outliers['SalePrice'].size} puntos rojos son outliers conocidos del dataset (ventas atípicas).")

In [ ]:
# --- Boxplot: Calidad General vs Precio ---
# OverallQual es una variable ordinal (escala 1-10) que califica la calidad de los materiales
# y acabados de la vivienda. Es la variable con mayor correlación con SalePrice (r=0.79).
# Un boxplot muestra cómo varía la distribución del precio en cada nivel de calidad,
# revelando medianas, rangos intercuartílicos y outliers por categoría.
plt.figure(figsize=(12, 6))
sns.boxplot(
    x=df['OverallQual'],
    y=df['SalePrice'],
    hue=df['OverallQual'],
    palette='viridis',
    legend=False
)
plt.title('Calidad General de Construcción vs Precio de Venta')
plt.xlabel('OverallQual — Calidad General (1=Muy Baja, 10=Muy Alta)')
plt.ylabel('SalePrice (USD)')
plt.tight_layout()
plt.savefig('visualization/boxplot_quality.png', dpi=100)
plt.close()

print("Correlación OverallQual ↔ SalePrice:", round(df['OverallQual'].corr(df['SalePrice']), 4))
print("--> A mayor calidad, mayor precio y mayor varianza (los rangos se amplían en niveles altos).")

### Análisis de Variables y Decisiones de Preprocesamiento

A partir del EDA, respondemos las preguntas clave del ejercicio:

#### ¿Hay variables con valores faltantes?
**Sí.** Hay 19 columnas con nulos. Las más críticas:
- `PoolQC` (99.5% nulos), `MiscFeature` (96.3%), `Alley` (93.8%), `Fence` (80.8%) — en estos casos el valor nulo **tiene significado**: la propiedad simplemente no tiene esa característica.
- `LotFrontage` (17.7%), `GarageYrBlt`, `GarageType`, etc. — nulos genuinamente faltantes.

**Estrategia**: Eliminar columnas con >40% de nulos, e imputar el resto con mediana (numéricas) o moda (categóricas).

#### ¿Hay variables categóricas a transformar?
**Sí.** 43 columnas son de tipo `str`/`object` (Neighborhood, SaleType, KitchenQual, etc.). Se convierten a numéricas mediante **One-Hot Encoding**, que crea una columna binaria por cada categoría posible y evita imponer un orden artificial.

#### ¿Hay variables con distribución sesgada u outliers?
**Sí.** `SalePrice` tiene un skewness de ~1.88 (sesgo positivo marcado). Varias variables de área también están sesgadas. Los modelos de árbol son robustos ante esto.

#### Elección del Modelo

| Criterio | Regresión Lineal | Random Forest | Gradient Boosting |
|----------|-----------------|---------------|-------------------|
| Variables mixtas (numéricas + categóricas) | Difícil | ✅ Excelente | ✅ Excelente |
| Relaciones no lineales | ❌ No captura | ✅ Captura | ✅ Captura |
| Outliers en target | ❌ Sensible | ✅ Robusto | ✅ Robusto |
| Multicolinealidad | ❌ Problemática | ✅ La maneja | ✅ La maneja |
| Interpretabilidad | Alta | Media | Media-Baja |
| Velocidad de entrenamiento | Muy rápida | Rápida | Lenta |

**Variables mixtas (numéricas + categóricas)** significa que el dataset contiene tanto variables numéricas como categóricas. Las numéricas ya están en formato que los modelos pueden procesar directamente, pero las categóricas son texto o etiquetas y deben convertirse a números. La forma estándar es One-Hot Encoding, que crea una columna binaria por cada categoría. En regresión lineal esto es problemático porque aumenta mucho el número de variables, lo que puede provocar sobreajuste y multicolinealidad. En modelos basados en árboles como Random Forest o Gradient Boosting esto no es un problema relevante porque manejan bien alta dimensionalidad y no dependen de relaciones lineales entre variables.

**Relaciones no lineales** significa que la relación entre una variable y el precio no sigue una línea que crece de manera lineal. Por ejemplo, pasar de calidad 2 a 3 puede tener poco impacto en el precio, pero pasar de 8 a 9 puede aumentarlo mucho. Los modelos de árboles dividen el espacio en regiones mediante reglas del tipo “si variable > umbral”, lo que les permite capturar relaciones complejas sin necesidad de transformaciones adicionales.

**Outliers en el target**  se refiere a valores extremos en la variable objetivo, como casas extremadamente caras o baratas. La regresión lineal minimiza el error cuadrático medio (MSE). Al elevar el error al cuadrado, los valores extremos tienen un impacto desproporcionado y pueden distorsionar el modelo. Los árboles son más robustos porque trabajan dividiendo los datos en rangos y un valor extremo no afecta de forma global a todas las predicciones.

**Multicolinealidad** significa que varias variables están altamente correlacionadas entre sí. Por ejemplo, dos variables que miden tamaño de la casa en distintas formas. En regresión lineal, si x₁ ≈ x₂ en la ecuación y = w₁x₁ + w₂x₂, el modelo no puede distinguir bien cuál de las dos variables es responsable del efecto, lo que hace que los coeficientes sean inestables y difíciles de interpretar. En modelos de árboles esto no es un problema porque el modelo selecciona automáticamente una de las variables en cada división y puede ignorar las redundantes.

**Interpretabilidad** se refiere a la facilidad para entender cómo el modelo toma decisiones. En regresión lineal cada coeficiente wᵢ indica directamente cuánto cambia la variable objetivo al cambiar la variable xᵢ, lo que hace el modelo muy interpretable. En Random Forest se pueden calcular importancias de variables, pero no existe una fórmula simple que explique cada predicción. En Gradient Boosting esto es aún más complejo porque el modelo combina muchos árboles de forma secuencial, por lo que es necesario usar técnicas adicionales como SHAP para interpretar resultados.

**Velocidad de entrenamiento** es el tiempo que tarda el modelo en ajustarse a los datos. La regresión lineal es muy rápida porque tiene una solución matemática directa o requiere una optimización simple. Random Forest es relativamente rápido porque entrena muchos árboles independientes que pueden ejecutarse en paralelo. Gradient Boosting es más lento porque construye los árboles de forma secuencial, donde cada árbol depende del anterior, lo que limita la paralelización y aumenta el tiempo de entrenamiento.

Como contexto adicional, todo esto está relacionado con el trade-off entre sesgo y varianza, donde los modelos lineales tienen alto sesgo pero baja varianza, mientras que los modelos de árboles reducen el sesgo a costa de mayor complejidad. También conecta con técnicas como regularización en modelos lineales para mitigar multicolinealidad y con métodos avanzados de interpretación como SHAP en modelos no lineales.

**Decisión**: Usaremos **Random Forest** como modelo base (paralelizable, robusto, sin transformar SalePrice) y **Gradient Boosting** como alternativa de mayor precisión para comparar.

In [ ]:
# Eliminar columnas con >40% de nulos (imputación sería poco fiable)
umbral_nulos = len(df) * 0.4
df_cleaned = df.dropna(thresh=len(df) - umbral_nulos, axis=1).copy()
columnas_eliminadas = set(df.columns) - set(df_cleaned.columns)
print(f"Columnas eliminadas por exceso de nulos (>40%): {sorted(columnas_eliminadas)}")

# Eliminar 'Id': identificador secuencial sin valor predictivo
if 'Id' in df_cleaned.columns:
    df_cleaned.drop('Id', axis=1, inplace=True)

# Separar features (X) y variable objetivo (y)
X = df_cleaned.drop('SalePrice', axis=1)
y = df_cleaned['SalePrice']

# Identificar tipos de columnas para aplicar transformaciones distintas
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"\nDimensiones tras limpieza: {df_cleaned.shape}")
print(f"Variables numéricas  : {len(numeric_features)}")
print(f"Variables categóricas: {len(categorical_features)}")
print(f"Total features usadas: {len(numeric_features) + len(categorical_features)}")

In [ ]:
# Usamos sklearn Pipeline para encadenar pasos de transformación de forma reproducible.
# La ventaja del Pipeline es que aplica el mismo preprocesamiento tanto al
# conjunto de entrenamiento como al de test, evitando data leakage.

# Transformador para variables NUMÉRICAS:
numeric_transformer = Pipeline(steps=[
    # SimpleImputer rellena los NaN con la MEDIANA de cada columna del train set.
    # Elegimos mediana (no media) porque es robusta ante outliers: un LotFrontage extremo
    # no distorsionará el valor de relleno.
    ('imputer', SimpleImputer(strategy='median')),
    # StandardScaler normaliza cada feature a media=0 y desviación=1.
    # Aunque Random Forest no lo necesita (los árboles son invariantes a escala),
    # lo incluimos para que el pipeline sirva también para modelos sensibles a escala
    # como el MLP o futuros modelos de regresión lineal.
    ('scaler', StandardScaler())
])

# Pipeline categórico: moda para nulos + OHE (handle_unknown ignora categorías no vistas en producción)
categorical_transformer = Pipeline(steps=[
    # Imputamos categóricas con la MODA (valor más frecuente) de cada columna.
    # Es la única opción con sentido semántico para variables nominales.
    ('imputer', SimpleImputer(strategy='most_frequent')),
    # OneHotEncoder transforma cada categoría en una columna binaria (0/1).
    # handle_unknown='ignore' descarta categorías no vistas durante el entrenamiento,
    # previniendo errores en producción cuando llegan datos nuevos.
    # sparse_output=False devuelve un array denso (compatible con todos los estimadores).
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# ColumnTransformer aplica transformadores distintos según el tipo de columna.
# Permite mantener un solo objeto que procesa todo el dataset de forma coordinada.
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# --- División Train/Test ---
# Usamos 80% para entrenar y 20% para evaluar.
# random_state=42 garantiza reproducibilidad (siempre la misma partición).
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Muestras de entrenamiento: {X_train.shape[0]}")
print(f"Muestras de evaluación  : {X_test.shape[0]}")
print("Pipeline de preprocesamiento construido.")

---
## EJERCICIO 2: Modelado y Evaluación

### Modelo Base: Random Forest Regressor

**¿Por qué Random Forest?**
- Es un ensamble de árboles de decisión entrenados sobre subconjuntos aleatorios del dataset (*bagging*).
- Cada árbol aprende una parte distinta del espacio de features, y el resultado final es el **promedio** de todas las predicciones individuales.
- Esto reduce el sobreajuste (overfitting) que sufre un árbol individual.
- No asume linealidad ni distribución normal en los datos.
- Maneja de forma nativa variables mixtas (numéricas + categóricas ya codificadas).

**Hiperparámetros iniciales:**
- `n_estimators=100`: 100 árboles en el bosque (balance velocidad/precisión)
- `random_state=42`: reproducibilidad

In [ ]:
# --- Construcción del Pipeline Completo: Preprocesamiento + Modelo ---
# Al encadenar el preprocesador y el regresor en un solo Pipeline:
# 1. fit() aprende las transformaciones y el modelo en un solo paso
# 2. predict() aplica exactamente las mismas transformaciones a datos nuevos
# 3. GridSearchCV puede buscar hiperparámetros tanto del preprocesamiento como del modelo
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
    # n_jobs=-1 usa todos los núcleos del CPU disponibles para entrenar en paralelo
])
print("Pipeline Random Forest creado.")

In [ ]:
# --- Entrenamiento del Modelo ---
# fit() ejecuta en secuencia:
#   1. Preprocesamiento: imputa nulos, escala numéricas, codifica categóricas (solo con train data)
#   2. Entrenamiento: construye los 100 árboles del Random Forest
# Medir el tiempo nos da información sobre la viabilidad de reentrenar el modelo en producción.
print("Entrenando Random Forest (modelo base)...")
start_time = time.time()
rf_pipeline.fit(X_train, y_train)
train_time = time.time() - start_time
print(f"Entrenamiento completado en {train_time:.2f} segundos.")

In [ ]:
y_pred = rf_pipeline.predict(X_test)

mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)          # RMSE: en la misma unidad que SalePrice (USD), más interpretable que MSE
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print("=" * 50)
print("  MÉTRICAS — Random Forest Base")
print("=" * 50)
print(f"  MSE  (Error Cuadrático Medio)   : ${mse:>15,.2f}")
print(f"  RMSE (Raíz del MSE)             : ${rmse:>15,.2f}")
print(f"  MAE  (Error Absoluto Medio)     : ${mae:>15,.2f}")
print(f"  R²   (Coeficiente Determinación): {r2:>18.4f}")
print("=" * 50)
print(f"\nInterpretación:")
print(f"  --> En promedio, el modelo se equivoca ~${mae:,.0f} por vivienda (MAE).")
print(f"  --> El R²={r2:.4f} indica que el modelo explica el {r2*100:.1f}% de la varianza del precio.")

# --- Validación Cruzada (k-fold) ---
# La validación cruzada evalúa el modelo sobre 5 particiones distintas del dataset,
# dándonos una estimación más robusta que un único test set.
# Un R² bajo o con alta desviación en CV indica que el modelo no generaliza bien.
print("\nValidación Cruzada (cv=5, métrica R²)...")
cv_scores = cross_val_score(rf_pipeline, X, y, cv=5, scoring='r2', n_jobs=-1)
print(f"  R² por fold: {[round(s, 4) for s in cv_scores]}")
print(f"  R² Medio   : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"  --> Baja desviación estándar indica que el modelo es estable entre particiones.")

### Análisis de Errores (Residuos)

Los residuos son la diferencia entre el valor real y el predicho: `e = y_real − y_pred`.

Un modelo ideal tendría residuos:
  - Distribuidos simétricamente alrededor de 0 (sin sesgo sistemático)
  - Con varianza constante para todos los rangos de precio (homocedasticidad)
  - Sin patrones (si hay patrones, el modelo no captura alguna relación)

In [ ]:
residuos = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Residuos vs Valores Predichos
# Si los puntos se dispersan aleatoriamente alrededor de la línea cero --> el modelo es correcto.
# Si hay un patrón en abanico (heterocedasticidad) --> los errores crecen con el precio.
axes[0].scatter(y_pred, residuos, alpha=0.4, color='mediumpurple')
axes[0].axhline(y=0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_title('Residuos vs Valores Predichos')
axes[0].set_xlabel('Precio Predicho (USD)')
axes[0].set_ylabel('Residuo = Real − Predicho (USD)')

# Gráfico 2: Distribución de los residuos
# Una distribución aproximadamente normal centrada en 0 es ideal.
# Colas largas indican predicciones muy erróneas en casos extremos.
sns.histplot(residuos, bins=40, kde=True, ax=axes[1], color='mediumpurple')
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_title('Distribución de los Residuos')
axes[1].set_xlabel('Residuo (USD)')

plt.suptitle('Análisis de Errores — Random Forest Base', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('visualization/residuos_rf.png', dpi=100)
plt.close()

print(f"Residuo medio (sesgo): ${residuos.mean():,.2f}  (debería ser ~0)")
print(f"Std de residuos      : ${residuos.std():,.2f}")
print(f"Residuo máximo (positivo): ${residuos.max():,.2f}")
print(f"Residuo mínimo (negativo): ${residuos.min():,.2f}")
print("\n--> Si hay heterocedasticidad (errores crecientes en precios altos), aplicar")
print("  log(SalePrice) como target reduciría ese patrón.")

---
## EJERCICIO 3: Optimización y Despliegue

### Ajuste de Hiperparámetros con GridSearchCV

El modelo base usa valores por defecto. **GridSearchCV** prueba sistemáticamente combinaciones de hiperparámetros y elige la que mejor R² obtiene en validación cruzada.

**Hiperparámetros a optimizar en Random Forest:**
- `n_estimators`: número de árboles. Más árboles → menor varianza, pero más tiempo de cómputo.
- `max_depth`: profundidad máxima de cada árbol. Árboles muy profundos → overfitting; muy superficiales → underfitting.

La notación `regressor__n_estimators` usa el doble guión bajo para referenciar el parámetro `n_estimators` del paso llamado `regressor` dentro del Pipeline.

#### Búsqueda de Hiperparámetros Óptimos [GridSearchCV]

- `cv=3`: validación cruzada de 3 folds (reducida para mantener tiempo razonable)
- `scoring='r2'`: optimizar el coeficiente de determinación
- `n_jobs=-1`: paralelizar en todos los núcleos disponibles

In [17]:

param_grid = {
    'regressor__n_estimators': [100, 200],    # Más árboles = mejor estimación, más lento
    'regressor__max_depth':    [10, 20, None] # None = árbol completo (sin límite de profundidad)
}

print(f"Combinaciones a probar: {len(param_grid['regressor__n_estimators']) * len(param_grid['regressor__max_depth'])} × 3 folds CV")
print("Iniciando búsqueda de hiperparámetros...\n")

start_gs = time.time()
grid_search = GridSearchCV(rf_pipeline, param_grid, cv=3, scoring='r2', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)
gs_time = time.time() - start_gs

best_rf = grid_search.best_estimator_
y_pred_best = best_rf.predict(X_test)
r2_rf_opt = r2_score(y_test, y_pred_best)
mae_rf_opt = mean_absolute_error(y_test, y_pred_best)
rmse_rf_opt = np.sqrt(mean_squared_error(y_test, y_pred_best))

print(f"\nMejores parámetros: {grid_search.best_params_}")
print(f"R² en CV (mejor)  : {grid_search.best_score_:.4f}")
print(f"\nResultados en Test Set:")
print(f"  R²  : {r2_rf_opt:.4f}  (base: {r2:.4f}, mejora: {r2_rf_opt - r2:+.4f})")
print(f"  MAE : ${mae_rf_opt:,.2f}  (base: ${mae:,.2f})")
print(f"  RMSE: ${rmse_rf_opt:,.2f}  (base: ${rmse:,.2f})")
print(f"\nTiempo total GridSearch: {gs_time:.1f} s")

Combinaciones a probar: 6 × 3 folds CV
Iniciando búsqueda de hiperparámetros...

Fitting 3 folds for each of 6 candidates, totalling 18 fits

Mejores parámetros: {'regressor__max_depth': None, 'regressor__n_estimators': 100}
R² en CV (mejor)  : 0.8572

Resultados en Test Set:
  R²  : 0.8921  (base: 0.8921, mejora: +0.0000)
  MAE : $17,605.26  (base: $17,605.26)
  RMSE: $28,764.70  (base: $28,764.70)

Tiempo total GridSearch: 2.9 s


### Modelo Alternativo: Gradient Boosting Regressor

A diferencia del Random Forest (que entrena árboles en paralelo e independientes), el Gradient Boosting entrena árboles de forma **secuencial**: cada árbol corrige los errores del anterior descendiendo el gradiente del error residual. Esto lo hace generalmente más preciso, pero más lento y más sensible a los hiperparámetros.

**Hiperparámetros usados:**
- `n_estimators=100`: número de rondas de boosting
- `learning_rate=0.1`: paso de descenso del gradiente (menor = más lento pero más preciso)

In [18]:
print("Entrenando Gradient Boosting Regressor...")
gb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42))
])

start_gb = time.time()
gb_pipeline.fit(X_train, y_train)
gb_train_time = time.time() - start_gb

y_pred_gb = gb_pipeline.predict(X_test)
r2_gb   = r2_score(y_test, y_pred_gb)
mae_gb  = mean_absolute_error(y_test, y_pred_gb)
rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))

print(f"Entrenamiento completado en {gb_train_time:.2f} s")
print(f"\nResultados Gradient Boosting en Test Set:")
print(f"  R²  : {r2_gb:.4f}")
print(f"  MAE : ${mae_gb:,.2f}")
print(f"  RMSE: ${rmse_gb:,.2f}")

Entrenando Gradient Boosting Regressor...
Entrenamiento completado en 0.48 s

Resultados Gradient Boosting en Test Set:
  R²  : 0.9068
  MAE : $16,627.72
  RMSE: $26,743.16


In [ ]:
# Comparamos los tres modelos entrenados: RF base, RF optimizado, y Gradient Boosting.
# El mejor modelo se elige por R² en el Test Set (no en Train, para evitar sesgo por overfitting).

resultados = {
    'Random Forest Base':      {'R²': r2,        'MAE': mae,        'RMSE': rmse,        'Tiempo (s)': train_time},
    'Random Forest Optimizado':{'R²': r2_rf_opt,  'MAE': mae_rf_opt, 'RMSE': rmse_rf_opt, 'Tiempo (s)': gs_time},
    'Gradient Boosting':       {'R²': r2_gb,      'MAE': mae_gb,     'RMSE': rmse_gb,     'Tiempo (s)': gb_train_time},
}

print("=" * 75)
print(f"{'Modelo':<28} {'R²':>8} {'MAE':>12} {'RMSE':>12} {'Tiempo':>10}")
print("=" * 75)
for nombre, m in resultados.items():
    print(f"{nombre:<28} {m['R²']:>8.4f} ${m['MAE']:>11,.0f} ${m['RMSE']:>11,.0f} {m['Tiempo (s)']:>8.1f}s")
print("=" * 75)

# Gráfico comparativo (inspirado en el estilo de modelos_casas.py)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

nombres = list(resultados.keys())
colores = ['steelblue', 'royalblue', 'seagreen']

r2_vals   = [resultados[n]['R²']  for n in nombres]
mae_vals  = [resultados[n]['MAE'] for n in nombres]
rmse_vals = [resultados[n]['RMSE'] for n in nombres]

axes[0].bar(nombres, r2_vals, color=colores, edgecolor='black')
axes[0].set_title('R² (Mayor es Mejor)')
axes[0].set_ylabel('R² Score')
axes[0].set_ylim(min(r2_vals) - 0.02, 1.0)
for i, v in enumerate(r2_vals):
    axes[0].text(i, v + 0.002, f'{v:.4f}', ha='center', fontsize=9, fontweight='bold')

axes[1].bar(nombres, mae_vals, color=colores, edgecolor='black')
axes[1].set_title('MAE — Error Absoluto Medio (Menor es Mejor)')
axes[1].set_ylabel('MAE (USD)')
for i, v in enumerate(mae_vals):
    axes[1].text(i, v + 200, f'${v:,.0f}', ha='center', fontsize=9, fontweight='bold')

axes[2].bar(nombres, rmse_vals, color=colores, edgecolor='black')
axes[2].set_title('RMSE — Raíz Error Cuadrático (Menor es Mejor)')
axes[2].set_ylabel('RMSE (USD)')
for i, v in enumerate(rmse_vals):
    axes[2].text(i, v + 200, f'${v:,.0f}', ha='center', fontsize=9, fontweight='bold')

for ax in axes:
    ax.set_xticklabels(nombres, rotation=12, ha='right')

plt.suptitle('Comparación de Modelos — Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('visualization/comparacion_modelos.png', dpi=100)
plt.close()

# Selección del modelo final
mejor_modelo_nombre = max(resultados, key=lambda n: resultados[n]['R²'])
print(f"\n--> Modelo seleccionado: '{mejor_modelo_nombre}' (R²={resultados[mejor_modelo_nombre]['R²']:.4f})")

if mejor_modelo_nombre == 'Gradient Boosting':
    modelo_final = gb_pipeline
else:
    modelo_final = best_rf

In [ ]:
# joblib serializa el objeto Pipeline completo (preprocesador + modelo) en un archivo binario.
# Al cargar el archivo en producción, el objeto recuperado ya incluye todos los parámetros
# aprendidos: medianas, escaladores, categorías OHE y pesos del modelo.
# Esto significa que para predecir basta con pasar los datos crudos sin preprocesar.
os.makedirs('model', exist_ok=True)
ruta_modelo = 'model/modelo_precio_casa.joblib'
joblib.dump(modelo_final, ruta_modelo)

print(f"Modelo guardado en: {ruta_modelo}")
print("Contenido del pipeline exportado:")
for nombre, paso in modelo_final.named_steps.items():
    print(f"  - {nombre}: {type(paso).__name__}")

### Despliegue como API REST con Flask

El archivo adjunto `app_proyecto_final.py` implementa una API REST con Flask que expone el modelo como servicio web. Esto permite que cualquier aplicación (frontend, app móvil, otro servicio) solicite predicciones enviando los datos de una vivienda en formato JSON.

**Arquitectura del despliegue:**

```
Cliente (JSON)  →  POST /predict  →  Flask API  →  Pipeline.predict()  →  { "precio_estimado_usd": 215000 }
```

**Endpoints disponibles:**
| Método | Ruta | Descripción |
|--------|------|-------------|
| GET | `/` | Estado del servicio y documentación |
| POST | `/predict` | Recibe datos de una casa, devuelve precio estimado |

**Ejemplo de uso (curl):**
```bash
curl -X POST http://localhost:5001/predict \
  -H "Content-Type: application/json" \
  -d '{"OverallQual": 7, "GrLivArea": 1500, "GarageCars": 2, "TotalBsmtSF": 900, "YearBuilt": 2000}'
```

**Para ejecutar la API:**
```bash
# 1. Asegurarse de haber ejecutado todas las celdas anteriores (genera el modelo)
# 2. Instalar Flask si no está disponible:
pip install flask
# 3. Ejecutar el servidor:
python app_proyecto_final.py
```

> El pipeline serializado con joblib maneja automáticamente el preprocesamiento de los datos de entrada, por lo que la API acepta datos crudos directamente sin necesidad de transformación previa.

---
## Conclusiones

### Resumen del Proceso

| Fase | Decisión | Justificación |
|------|----------|---------------|
| EDA | Análisis de nulos, distribución y correlación | Identificar qué datos necesitan tratamiento antes de modelar |
| Filtrado | Eliminar columnas con >40% nulos + `Id` | Evitar ruido y correlaciones espurias |
| Imputación | Mediana para numéricos, moda para categóricos | Robustez ante outliers / semántica correcta para categorías |
| Encoding | One-Hot Encoding | Convierte categorías sin imponer un orden artificial |
| Modelo base | Random Forest (100 árboles) | Robusto, paralelizable, no asume linealidad |
| Optimización | GridSearchCV (n_estimators, max_depth) | Búsqueda sistemática del mejor balance bias-varianza |
| Comparación | Random Forest vs Gradient Boosting | Gradient Boosting tiende a superar a RF en precisión a costa de tiempo |
| Despliegue | Flask API + joblib | El Pipeline serializado incluye el preprocesamiento, simplificando la API |

### Lecciones Clave

1. **Los modelos de árbol son ideales para este tipo de dataset**: mezcla de variables numéricas y categóricas, distribuciones sesgadas, y relaciones no lineales.
2. **El Pipeline de sklearn es fundamental**: previene *data leakage* (las transformaciones se aprenden solo con datos de entrenamiento) y hace el modelo directamente desplegable.
3. **Gradient Boosting supera a Random Forest en precisión** pero a mayor costo computacional: el boosting secuencial no se puede paralelizar de la misma forma.
4. **El análisis de residuos revela heterocedasticidad**: los errores son mayores para viviendas de precio alto, lo que sugiere que aplicar `log(SalePrice)` como target podría mejorar aún más el modelo.